# Benchmark the complete Lily dataset

This notebook benchmarks parity-game solvers and forced multistrategies. The final section lets you select any of the paper datasets and algorithms, including Chain-MS, then generates three aggregate tables from the selected results. It reports per-game measurements and aggregate statistics. Timing is run without `tracemalloc` so memory tracing does not distort runtime; BDD node counts are reported for FPJ.

The generic runners also support one in-memory game (`benchmark_game`), any named benchmark set (`benchmark_dataset`), or all seven paper datasets (`benchmark_all_datasets`). Dataset runs use four worker processes by default and print a completion line as each solver run finishes; pass `workers=1` for sequential execution, `progress=False` to silence updates, or a callback to collect structured `BenchmarkProgress` events. Parallel runs use more peak memory, while each worker releases solver state between tasks. Custom methods can be passed as a `{name: solver_callable}` mapping.

In [1]:
from pathlib import Path
import sys

search_roots = (Path.cwd(), *Path.cwd().parents)
CODE = next(
    (candidate for root in search_roots
     for candidate in (root, root / 'code')
     if (candidate / 'src' / 'parity_bdd_lab').is_dir()),
    None,
)
if CODE is None:
    raise FileNotFoundError(f'Could not locate the code project from {Path.cwd()}')
sys.path.insert(0, str(CODE / 'src'))

import pandas as pd
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
from parity_bdd_lab import benchmark_dataset, records_to_dataframe

WORKERS = 4

## Run every Lily game

Set `REPEATS` in the next cell to control how many records are collected per game/method pair; the notebook summarizes repeats by mean and standard deviation. Parallel progress is reported as runs finish; results are returned in dataset order. Memory tracing is optional and disabled here.

In [ ]:
REPEATS = 3
METHODS = ('DFI', 'DFI-ns', 'FPJ', 'Zielonka', 'Chain-MS')

run_records = benchmark_dataset(
    'Lily',
    methods=METHODS,
    repeats=REPEATS,
    measure_memory=False,
    workers=WORKERS,
)
run_results = records_to_dataframe(run_records)
assert run_results['game'].nunique() == 23
assert len(run_results) == 23 * len(METHODS) * REPEATS

# Check that the methods agree on every run, then aggregate runtimes per game/method.
agreement = run_results.groupby('game')['solution_fingerprint'].nunique()
assert (agreement == 1).all(), agreement[agreement != 1]
results = (
    run_results.groupby(['game', 'method'], as_index=False)
    .agg(
        mean_seconds=('runtime_seconds', 'mean'),
        std_seconds=('runtime_seconds', 'std'),
        even_winning_vertices=('even_winning_vertices', 'first'),
        odd_winning_vertices=('odd_winning_vertices', 'first'),
        strategy_edges=('strategy_edges', 'first'),
        bdd_nodes=('bdd_nodes', 'first'),
        solution_fingerprint=('solution_fingerprint', 'first'),
    )
)
display(results.sort_values(['game', 'method']).reset_index(drop=True))

[1/345 runs; game 1/23] Lily / lilydemo01.tlsf.ehoa.oink / DFI run 1/3 complete
[2/345 runs; game 1/23] Lily / lilydemo01.tlsf.ehoa.oink / DFI run 2/3 complete
[3/345 runs; game 1/23] Lily / lilydemo01.tlsf.ehoa.oink / DFI-ns run 1/3 complete
[4/345 runs; game 1/23] Lily / lilydemo01.tlsf.ehoa.oink / DFI run 3/3 complete
[5/345 runs; game 1/23] Lily / lilydemo01.tlsf.ehoa.oink / DFI-ns run 2/3 complete
[6/345 runs; game 1/23] Lily / lilydemo01.tlsf.ehoa.oink / DFI-ns run 3/3 complete
[7/345 runs; game 1/23] Lily / lilydemo01.tlsf.ehoa.oink / FPJ run 2/3 complete
[8/345 runs; game 1/23] Lily / lilydemo01.tlsf.ehoa.oink / FPJ run 1/3 complete
[9/345 runs; game 1/23] Lily / lilydemo01.tlsf.ehoa.oink / Zielonka run 3/3 complete
[10/345 runs; game 1/23] Lily / lilydemo01.tlsf.ehoa.oink / Zielonka run 1/3 complete
[11/345 runs; game 1/23] Lily / lilydemo01.tlsf.ehoa.oink / FPJ run 3/3 complete
[12/345 runs; game 1/23] Lily / lilydemo01.tlsf.ehoa.oink / Zielonka run 2/3 complete
[13/345 runs;

AssertionError: 

## Aggregate by method

These totals sum the five-run means over all Lily games and show the median and maximum per-game mean runtime.

In [ ]:
method_summary = (
    results.groupby('method', as_index=False)
    .agg(
        games=('game', 'nunique'),
        total_mean_seconds=('mean_seconds', 'sum'),
        median_game_mean_seconds=('mean_seconds', 'median'),
        max_game_mean_seconds=('mean_seconds', 'max'),
        total_even_wins=('even_winning_vertices', 'sum'),
        total_odd_wins=('odd_winning_vertices', 'sum'),
        total_strategy_edges=('strategy_edges', 'sum'),
    )
)
display(method_summary)

## Other scopes

The same runner can benchmark one game or all datasets. Uncomment and run one of these when needed; benchmarking all datasets may take substantially longer.

In [ ]:
# One game already loaded as a ParityGame:
# from parity_bdd_lab import benchmark_game, read_pg
# game_records = benchmark_game(read_pg(CODE / 'data/games/image_example.pg',
#                                    vertex_labels=dict(enumerate('abcdefghi')),
#                                    name='image_example'),
#                                methods=('DFI', 'DFI-ns', 'FPJ', 'Zielonka'))

# All seven benchmark datasets, one game loaded at a time:
from parity_bdd_lab import benchmark_all_datasets
all_records = benchmark_all_datasets(methods=('DFI', 'DFI-ns', 'FPJ', 'Zielonka', 'Chain-MS'), repeats=1, workers=WORKERS)
all_results = records_to_dataframe(all_records)

In [ ]:
all_results

## Select datasets and methods for the comparison

Choose one dataset, a list of datasets, or `None` for all seven. `Chain-MS` uses an independent chain-quotient solver and constructs its own cycle-based justification graph. Its node count is for BDD relations over chain IDs; each ID maps to a vertex tuple. FPJ and the other symbolic solvers report their own BDD nodes.


In [ ]:
from parity_bdd_lab import benchmark_selection, benchmark_dataset_names, records_to_dataframe

WORKERS = globals().get('WORKERS', 4)

BENCHMARK_DATASETS = None  # e.g. 'Lily' or ['Lily', 'AMBA']; None selects all
BENCHMARK_METHODS = ['FPJ', 'Chain-MS']  # or include DFI, DFI-ns, Zielonka
BENCHMARK_REPEATS = 1

selected_records = benchmark_selection(
    BENCHMARK_DATASETS,
    methods=BENCHMARK_METHODS,
    repeats=BENCHMARK_REPEATS,
    workers=WORKERS,
)
all_results = records_to_dataframe(selected_records)
_available = benchmark_dataset_names()
_dataset_names = {name.casefold(): name for name in _available}
_requested = _available if BENCHMARK_DATASETS is None else (
    [BENCHMARK_DATASETS] if isinstance(BENCHMARK_DATASETS, str) else list(BENCHMARK_DATASETS)
)
ACTIVE_DATASETS = [_dataset_names[name.casefold()] for name in _requested]
METHOD_ORDER = list(BENCHMARK_METHODS)
display(all_results)


## Paper-style dataset tables

The following cells use the existing `all_results` dataframe; they do not rerun the solvers. Table 1 reads each benchmark game to calculate graph statistics and the static game-encoding BDD size.


In [ ]:
import gc
from parity_bdd_lab import benchmark_dataset_names, collect_bdd_garbage, iter_benchmark_files, read_pg
from parity_bdd_lab.fpj import _Encoding

# One row per game. Games and BDD managers are released after measuring each file.
game_statistics = []
for dataset in ACTIVE_DATASETS:
    for game_path in iter_benchmark_files(dataset):
        game = read_pg(game_path)
        vertices = game.vertices
        priorities = tuple(game.priority[v] for v in vertices)
        vertex_count = len(vertices)
        edge_count = sum(len(game.successors[v]) for v in vertices)

        encoding = _Encoding(game)
        collect_bdd_garbage(encoding.bdd)
        static_bdd_nodes = len(encoding.bdd)
        game_statistics.append({
            'dataset': dataset,
            'game': game_path.name,
            'vertices': vertex_count,
            'edges': edge_count,
            'min priority': min(priorities),
            'max priority': max(priorities),
            'static BDD nodes': static_bdd_nodes,
        })
        del encoding, game
        gc.collect()

game_statistics = pd.DataFrame(game_statistics)
dataset_summary_rows = []
for dataset in ACTIVE_DATASETS:
    games_in_set = game_statistics[game_statistics['dataset'] == dataset]
    dataset_summary_rows.append({
        'Set': dataset,
        '#games': games_in_set['game'].nunique(),
        'avg. n': games_in_set['vertices'].mean(),
        'max. n': games_in_set['vertices'].max(),
        'priorities': (
            f"{games_in_set['min priority'].min()}–{games_in_set['max priority'].max()}"
        ),
        'avg. out-degree': games_in_set['edges'].sum() / games_in_set['vertices'].sum(),
        'avg. #BDD nodes': games_in_set['static BDD nodes'].mean(),
    })

dataset_summary = pd.DataFrame(dataset_summary_rows)
display(dataset_summary.round({
    'avg. n': 0,
    'avg. out-degree': 2,
    'avg. #BDD nodes': 0,
}))


### Cumulative runtime by dataset

Runtime is summed over all games within each repeat, then averaged across repeats. With a single run, the result is the cumulative time for that run.


In [ ]:
dataset_order = ACTIVE_DATASETS
method_order = METHOD_ORDER

per_repeat_runtime = (
    all_results
    .groupby(['dataset', 'method', 'run'], as_index=False)['runtime_seconds']
    .sum()
)
runtime_by_dataset = (
    per_repeat_runtime
    .groupby(['dataset', 'method'])['runtime_seconds']
    .mean()
    .unstack('method')
    .reindex(index=dataset_order, columns=method_order)
    .rename(columns={name: name.casefold() for name in method_order})
)
display(runtime_by_dataset.round(2))


### Average decision diagram nodes by dataset

This uses the `bdd_nodes` values already stored in `all_results`: BDD nodes for DFI, DFI-ns, and FPJ, and chain-relation BDD nodes for Chain-MS. These are live manager sizes when each solver returns, not peak node counts. Zielonka does not use a decision diagram and appears blank.


In [ ]:
bdd_node_summary = (
    all_results
    .groupby(['dataset', 'method'])['bdd_nodes']
    .mean()
    .unstack('method')
    .reindex(index=dataset_order, columns=method_order)
    .rename(columns={name: name.casefold() for name in method_order})
)
display(bdd_node_summary.round(0))
